# 07 — Portfolio Visuals

Three publication-ready charts:
1. Bracket diagram — full tournament with win probabilities
2. Championship probability — each team's path-to-title %
3. Model performance card — AUC, SHAP features, accuracy by round

In [ ]:
import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from matplotlib.patches import FancyBboxPatch

sys.path.insert(0, str(Path().resolve().parent))
from src.models import load_model, train_test_split_by_season

OUT_DIR = Path().resolve().parent
RAW_DIR = Path().resolve().parent / "data" / "raw"
PROC_DIR = Path().resolve().parent / "data" / "processed"

In [ ]:
# ── load model + 2025-26 metrics ────────────────────────────
pipeline = load_model("series_xgboost")

metrics = pd.read_parquet(RAW_DIR / "team_metrics_2025-26.parquet")
eff = metrics.set_index("TEAM_ID")

hist_games = pd.concat(
    [
        pd.read_parquet(RAW_DIR / f"playoff_games_{s}.parquet").assign(season=s)
        for s in ["2021-22", "2022-23", "2023-24"]
    ],
    ignore_index=True,
)
hist_games["round"] = hist_games["GAME_ID"].str[6:8].astype(int)
hist_games["win"] = (hist_games["WL"] == "W").astype(int)
hist_games["is_finals"] = (hist_games["round"] == 4).astype(int)
season_stats = (
    hist_games.groupby(["season", "TEAM_ID"])
    .agg(
        playoff_wins=("win", "sum"),
        playoff_games=("win", "count"),
        reached_finals=("is_finals", "max"),
    )
    .reset_index()
)
season_stats["playoff_win_pct"] = (
    season_stats["playoff_wins"] / season_stats["playoff_games"]
)
season_stats["season_year"] = season_stats["season"].str[:4].astype(int)


def prior_stats(team_id, current_year=2025):
    hist = season_stats[season_stats["TEAM_ID"] == team_id]
    p3 = hist[hist["season_year"] >= current_year - 3]
    p5 = hist[hist["season_year"] >= current_year - 5]
    return (
        round(p3["playoff_win_pct"].mean(), 3) if len(p3) > 0 else 0.5,
        int(p5["reached_finals"].sum()),
    )


FEATURE_COLS = [
    "home_ortg",
    "away_ortg",
    "home_drtg",
    "away_drtg",
    "home_net_rtg",
    "away_net_rtg",
    "net_rtg_diff",
    "home_pace",
    "away_pace",
    "ortg_diff",
    "drtg_diff",
    "home_win_pct",
    "away_win_pct",
    "win_pct_diff",
    "home_oreb_pct",
    "away_oreb_pct",
    "home_tov_pct",
    "away_tov_pct",
    "home_playoff_win_pct_3yr",
    "away_playoff_win_pct_3yr",
    "playoff_win_pct_diff",
    "home_finals_apps_5yr",
    "away_finals_apps_5yr",
]


def build_row(h_name, h_id, a_name, a_id):
    h = eff.loc[h_id]
    a = eff.loc[a_id]
    h_hist, h_fin = prior_stats(h_id)
    a_hist, a_fin = prior_stats(a_id)
    return {
        "matchup": f"{h_name} vs {a_name}",
        "home_ortg": h["E_OFF_RATING"],
        "away_ortg": a["E_OFF_RATING"],
        "home_drtg": h["E_DEF_RATING"],
        "away_drtg": a["E_DEF_RATING"],
        "home_net_rtg": h["E_NET_RATING"],
        "away_net_rtg": a["E_NET_RATING"],
        "net_rtg_diff": h["E_NET_RATING"] - a["E_NET_RATING"],
        "home_pace": h["E_PACE"],
        "away_pace": a["E_PACE"],
        "ortg_diff": h["E_OFF_RATING"] - a["E_OFF_RATING"],
        "drtg_diff": h["E_DEF_RATING"] - a["E_DEF_RATING"],
        "home_win_pct": h["W_PCT"],
        "away_win_pct": a["W_PCT"],
        "win_pct_diff": h["W_PCT"] - a["W_PCT"],
        "home_oreb_pct": h["E_OREB_PCT"],
        "away_oreb_pct": a["E_OREB_PCT"],
        "home_tov_pct": h["E_TM_TOV_PCT"],
        "away_tov_pct": a["E_TM_TOV_PCT"],
        "home_playoff_win_pct_3yr": h_hist,
        "away_playoff_win_pct_3yr": a_hist,
        "playoff_win_pct_diff": h_hist - a_hist,
        "home_finals_apps_5yr": h_fin,
        "away_finals_apps_5yr": a_fin,
    }


def predict_matchup(h_name, h_id, a_name, a_id):
    row = build_row(h_name, h_id, a_name, a_id)
    df = pd.DataFrame([row])
    return pipeline.predict_proba(df[FEATURE_COLS])[0, 1]


def advance(h_name, h_id, a_name, a_id, prob):
    return (h_name, h_id) if prob >= 0.5 else (a_name, a_id)


def order_by_seed(team_a, team_b):
    return (
        (team_a, team_b)
        if eff.loc[team_a[1], "W_PCT"] >= eff.loc[team_b[1], "W_PCT"]
        else (team_b, team_a)
    )


# ── run all three rounds ─────────────────────────────────────
R2 = [
    ("New York Knicks", 1610612752, "Philadelphia 76ers", 1610612755),
    ("Detroit Pistons", 1610612765, "Cleveland Cavaliers", 1610612739),
    ("Oklahoma City Thunder", 1610612760, "Los Angeles Lakers", 1610612747),
    ("San Antonio Spurs", 1610612759, "Minnesota Timberwolves", 1610612750),
]
r2_probs = [predict_matchup(*m) for m in R2]

r2_winners = [advance(*R2[i], r2_probs[i]) for i in range(4)]
east_h, east_a = order_by_seed(r2_winners[0], r2_winners[1])
west_h, west_a = order_by_seed(r2_winners[2], r2_winners[3])

CF = [(*east_h, *east_a), (*west_h, *west_a)]
cf_probs = [predict_matchup(*m) for m in CF]
cf_winners = [advance(*CF[i], cf_probs[i]) for i in range(2)]

fin_h, fin_a = order_by_seed(cf_winners[0], cf_winners[1])
FINALS = [(*fin_h, *fin_a)]
fin_prob = predict_matchup(*FINALS[0])
champion = advance(*FINALS[0], fin_prob)

print("R2 probs:    ", [f"{p:.0%}" for p in r2_probs])
print("CF probs:    ", [f"{p:.0%}" for p in cf_probs])
print("Finals prob: ", f"{fin_prob:.0%}")
print("Champion:    ", champion[0])

## 1. Bracket Diagram

In [ ]:
TEAM_SHORT = {
    "New York Knicks": "Knicks",
    "Philadelphia 76ers": "76ers",
    "Detroit Pistons": "Pistons",
    "Cleveland Cavaliers": "Cavs",
    "Oklahoma City Thunder": "OKC",
    "Los Angeles Lakers": "Lakers",
    "San Antonio Spurs": "Spurs",
    "Minnesota Timberwolves": "Wolves",
}


def sn(name):
    return TEAM_SHORT.get(name, name.split()[-1])


# ── layout constants ─────────────────────────────────────────
# 8 team slots: East top (y 5–8), East bottom (y 3–5 gap), West (y 0–4)
# Using y-positions with a small gap between conferences
CONF_GAP = 0.6

# R2 team y-positions: pairs are (7.5,6.5), (5.5,4.5) | (3.5,2.5), (1.5,0.5)
R2_Y = [
    [7.5, 6.5],  # East matchup 0
    [5.5, 4.5],  # East matchup 1
    [3.0, 2.0],  # West matchup 0  (gap between conferences)
    [1.0, 0.0],  # West matchup 1
]
CF_Y = [(7.0, 5.0), (2.5, 0.5)]  # mid-point of each R2 pair
FIN_Y = 3.875  # midpoint of both CF results

X_R2, X_CF, X_FIN, X_CHAMP = 0, 3.5, 6.5, 8.8
BOX_W, BOX_H = 2.8, 0.55

COLORS = {
    "win_box": "#1a3a5c",
    "lose_box": "#d0d0d0",
    "win_txt": "white",
    "lose_txt": "#666666",
    "line": "#555555",
    "conf_lbl": "#888888",
    "bg": "#f7f7f7",
}

fig, ax = plt.subplots(figsize=(17, 9))
fig.patch.set_facecolor(COLORS["bg"])
ax.set_facecolor(COLORS["bg"])
ax.set_xlim(-0.2, 10.2)
ax.set_ylim(-0.6, 8.7)
ax.axis("off")
fig.suptitle(
    "2025–26 NBA Playoffs — Bracket Projection",
    fontsize=15,
    fontweight="bold",
    y=0.97,
    color="#1a1a1a",
)


def draw_box(ax, x, y_center, label, prob_label="", is_winner=True, fontsize=9.5):
    color = COLORS["win_box"] if is_winner else COLORS["lose_box"]
    tcolor = COLORS["win_txt"] if is_winner else COLORS["lose_txt"]
    rect = FancyBboxPatch(
        (x - BOX_W / 2, y_center - BOX_H / 2),
        BOX_W,
        BOX_H,
        boxstyle="round,pad=0.04",
        linewidth=0,
        facecolor=color,
        zorder=3,
    )
    ax.add_patch(rect)
    # team name
    ax.text(
        x - BOX_W / 2 + 0.12,
        y_center,
        label,
        va="center",
        ha="left",
        fontsize=fontsize,
        fontweight="bold" if is_winner else "normal",
        color=tcolor,
        zorder=4,
    )
    # prob badge on right side
    if prob_label and is_winner:
        ax.text(
            x + BOX_W / 2 - 0.12,
            y_center,
            prob_label,
            va="center",
            ha="right",
            fontsize=8,
            color="#aaccff",
            zorder=4,
        )


def connect(ax, x_from, y_from, x_to, y_to, color="#555555", lw=1.5):
    x_mid = (x_from + x_to) / 2
    ax.plot(
        [x_from, x_mid, x_mid, x_to],
        [y_from, y_from, y_to, y_to],
        color=color,
        lw=lw,
        zorder=1,
        solid_capstyle="round",
    )


# ── column headers ───────────────────────────────────────────
for label, x in [
    ("SECOND ROUND", X_R2),
    ("CONF FINALS", X_CF),
    ("NBA FINALS", X_FIN),
    ("CHAMPION", X_CHAMP),
]:
    ax.text(
        x,
        8.45,
        label,
        ha="center",
        va="center",
        fontsize=8.5,
        color="#555555",
        fontweight="bold",
        bbox=dict(facecolor="#e0e0e0", edgecolor="none", boxstyle="round,pad=0.3"),
    )

# conference labels
ax.text(
    -0.1,
    6.5,
    "EAST",
    ha="center",
    va="center",
    fontsize=8,
    color=COLORS["conf_lbl"],
    rotation=90,
    fontweight="bold",
)
ax.text(
    -0.1,
    2.0,
    "WEST",
    ha="center",
    va="center",
    fontsize=8,
    color=COLORS["conf_lbl"],
    rotation=90,
    fontweight="bold",
)

# ── R2 boxes ─────────────────────────────────────────────────
r2_winner_names = [advance(*R2[i], r2_probs[i])[0] for i in range(4)]
for mi, matchup in enumerate(R2):
    h_name, h_id, a_name, a_id = matchup
    prob = r2_probs[mi]
    h_wins = prob >= 0.5
    y_h, y_a = R2_Y[mi]
    draw_box(ax, X_R2, y_h, sn(h_name), f"{prob:.0%}", is_winner=h_wins)
    draw_box(ax, X_R2, y_a, sn(a_name), f"{1 - prob:.0%}", is_winner=not h_wins)
    # connector between the two R2 teams
    ax.plot(
        [
            X_R2 + BOX_W / 2,
            X_R2 + BOX_W / 2 + 0.25,
            X_R2 + BOX_W / 2 + 0.25,
            X_R2 + BOX_W / 2,
        ],
        [y_h, y_h, y_a, y_a],
        color="#bbbbbb",
        lw=1,
        zorder=1,
    )

# ── CF boxes ─────────────────────────────────────────────────
cf_matchups_draw = [(*east_h, *east_a), (*west_h, *west_a)]
cf_winner_names = [advance(*cf_matchups_draw[i], cf_probs[i])[0] for i in range(2)]
cf_y_centers = [(R2_Y[0][0] + R2_Y[1][1]) / 2, (R2_Y[2][0] + R2_Y[3][1]) / 2]
cf_gap = (R2_Y[0][0] - R2_Y[1][1]) / 2 * 0.7

for ci, (matchup, prob, yc) in enumerate(zip(cf_matchups_draw, cf_probs, cf_y_centers)):
    h_name, h_id, a_name, a_id = matchup
    h_wins = prob >= 0.5
    y_h = yc + 0.55
    y_a = yc - 0.55
    draw_box(ax, X_CF, y_h, sn(h_name), f"{prob:.0%}", is_winner=h_wins)
    draw_box(ax, X_CF, y_a, sn(a_name), f"{1 - prob:.0%}", is_winner=not h_wins)
    ax.plot(
        [
            X_CF + BOX_W / 2,
            X_CF + BOX_W / 2 + 0.25,
            X_CF + BOX_W / 2 + 0.25,
            X_CF + BOX_W / 2,
        ],
        [y_h, y_h, y_a, y_a],
        color="#bbbbbb",
        lw=1,
        zorder=1,
    )
    # lines from R2 winner to CF
    for r2_mi, cf_team in [(ci * 2, h_name), (ci * 2 + 1, a_name)]:
        r2_winner = r2_winner_names[r2_mi]
        r2_y_top, r2_y_bot = R2_Y[r2_mi]
        r2_winner_y = r2_y_top if r2_winner == R2[r2_mi][0] else r2_y_bot
        cf_y = y_h if cf_team == h_name else y_a
        connect(
            ax,
            X_R2 + BOX_W / 2,
            r2_winner_y,
            X_CF - BOX_W / 2,
            cf_y,
            color="#1a3a5c" if r2_winner == cf_team else "#cccccc",
            lw=1.6,
        )

# ── Finals box ───────────────────────────────────────────────
fin_h_name, fin_h_id, fin_a_name, fin_a_id = FINALS[0]
fin_h_wins = fin_prob >= 0.5
y_fin_h = FIN_Y + 0.55
y_fin_a = FIN_Y - 0.55
draw_box(
    ax,
    X_FIN,
    y_fin_h,
    sn(fin_h_name),
    f"{fin_prob:.0%}",
    is_winner=fin_h_wins,
    fontsize=10,
)
draw_box(
    ax,
    X_FIN,
    y_fin_a,
    sn(fin_a_name),
    f"{1 - fin_prob:.0%}",
    is_winner=not fin_h_wins,
    fontsize=10,
)
ax.plot(
    [
        X_FIN + BOX_W / 2,
        X_FIN + BOX_W / 2 + 0.25,
        X_FIN + BOX_W / 2 + 0.25,
        X_FIN + BOX_W / 2,
    ],
    [y_fin_h, y_fin_h, y_fin_a, y_fin_a],
    color="#bbbbbb",
    lw=1,
    zorder=1,
)

# lines from CF winner to Finals
for ci, cf_winner_name in enumerate(cf_winner_names):
    yc = cf_y_centers[ci]
    cf_h_name = cf_matchups_draw[ci][0]
    cf_winner_y = yc + 0.55 if cf_winner_name == cf_h_name else yc - 0.55
    fin_y = y_fin_h if cf_winner_name == fin_h_name else y_fin_a
    connect(
        ax,
        X_CF + BOX_W / 2,
        cf_winner_y,
        X_FIN - BOX_W / 2,
        fin_y,
        color="#1a3a5c",
        lw=2.0,
    )

# ── Champion box ─────────────────────────────────────────────
champ_y_src = y_fin_h if champion[0] == fin_h_name else y_fin_a
connect(
    ax, X_FIN + BOX_W / 2, champ_y_src, X_CHAMP - 1.0, FIN_Y, color="#c8a000", lw=2.5
)
champ_rect = FancyBboxPatch(
    (X_CHAMP - 1.0, FIN_Y - 0.75),
    2.0,
    1.5,
    boxstyle="round,pad=0.08",
    linewidth=2,
    facecolor="#c8a000",
    edgecolor="#9a7800",
    zorder=3,
)
ax.add_patch(champ_rect)
ax.text(
    X_CHAMP,
    FIN_Y + 0.25,
    sn(champion[0]),
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color="white",
    zorder=4,
)
ax.text(
    X_CHAMP,
    FIN_Y - 0.25,
    "★  Projected",
    ha="center",
    va="center",
    fontsize=8,
    color="#fff3cc",
    zorder=4,
)

# source note
ax.text(
    5.0,
    -0.5,
    "XGBoost series model · AUC 0.683 · metrics from Basketball Reference 2025–26",
    ha="center",
    va="center",
    fontsize=7.5,
    color="#999999",
    style="italic",
)

plt.tight_layout()
out = OUT_DIR / "bracket_2025-26.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=COLORS["bg"])
plt.show()
print(f"Saved → {out}")

## 2. Championship Probability

In [ ]:
# P(team wins championship) = product along their bracket path.
# For each team, compute conditional on them winning each round.
# We need P(team wins R2) × P(team wins CF | reached CF) × P(team wins Finals | reached Finals)

all_teams = [
    # (name, id, r2_matchup_idx, is_higher_seed_in_r2)
    ("New York Knicks", 1610612752, 0, True),
    ("Philadelphia 76ers", 1610612755, 0, False),
    ("Detroit Pistons", 1610612765, 1, True),
    ("Cleveland Cavaliers", 1610612739, 1, False),
    ("Oklahoma City Thunder", 1610612760, 2, True),
    ("Los Angeles Lakers", 1610612747, 2, False),
    ("San Antonio Spurs", 1610612759, 3, True),
    ("Minnesota Timberwolves", 1610612750, 3, False),
]


def p_win_r2(name, r2_idx, is_higher):
    p = r2_probs[r2_idx]
    return p if is_higher else 1 - p


def p_win_cf_given_r2(name, team_id, r2_idx, is_higher):
    """Simulate all possible CF opponents (both possible R2 winners from the other R2 slot)."""
    # Which R2 slot provides the CF opponent?
    if r2_idx in [0, 1]:  # East
        opp_r2_idx = 1 if r2_idx == 0 else 0
    else:  # West
        opp_r2_idx = 3 if r2_idx == 2 else 2
    opp_h_name, opp_h_id, opp_a_name, opp_a_id = R2[opp_r2_idx]
    p_opp_higher = r2_probs[opp_r2_idx]
    # weight CF win prob by probability of facing each opponent
    p_total = 0.0
    for opp_name, opp_id, p_opp in [
        (opp_h_name, opp_h_id, p_opp_higher),
        (opp_a_name, opp_a_id, 1 - p_opp_higher),
    ]:
        # who hosts CF?
        if eff.loc[team_id, "W_PCT"] >= eff.loc[opp_id, "W_PCT"]:
            p_cf = predict_matchup(name, team_id, opp_name, opp_id)
        else:
            p_cf = 1 - predict_matchup(opp_name, opp_id, name, team_id)
        p_total += p_opp * p_cf
    return p_total


def p_win_finals_given_cf(name, team_id, r2_idx):
    """Simulate all possible Finals opponents (4 teams from the other conference)."""
    if r2_idx in [0, 1]:  # East → face West teams
        opp_r2_idxs = [2, 3]
    else:  # West → face East teams
        opp_r2_idxs = [0, 1]
    # enumerate all 4 possible opponents (2 per R2 slot × 2 slots)
    opp_candidates = []
    for oi in opp_r2_idxs:
        oh, ohid, oa, oaid = R2[oi]
        pop = r2_probs[oi]
        # probability of facing this team in finals = p_win_r2 × p_win_cf
        for oname, oid, p_reach_r2 in [(oh, ohid, pop), (oa, oaid, 1 - pop)]:
            # simplified: use marginal probability of this opponent reaching finals
            p_cf_opp = p_win_cf_given_r2(oname, oid, oi, oname == oh)
            opp_candidates.append((oname, oid, p_reach_r2 * p_cf_opp))
    # normalize
    total_w = sum(x[2] for x in opp_candidates)
    p_finals = 0.0
    for opp_name, opp_id, w in opp_candidates:
        w_norm = w / total_w
        if eff.loc[team_id, "W_PCT"] >= eff.loc[opp_id, "W_PCT"]:
            p_fin = predict_matchup(name, team_id, opp_name, opp_id)
        else:
            p_fin = 1 - predict_matchup(opp_name, opp_id, name, team_id)
        p_finals += w_norm * p_fin
    return p_finals


results = []
for name, tid, r2_idx, is_higher in all_teams:
    pr2 = p_win_r2(name, r2_idx, is_higher)
    pcf = p_win_cf_given_r2(name, tid, r2_idx, is_higher)
    pfin = p_win_finals_given_cf(name, tid, r2_idx)
    champ_p = pr2 * pcf * pfin
    results.append(
        {
            "team": name,
            "short": sn(name),
            "p_r2": pr2,
            "p_cf": pcf,
            "p_fin": pfin,
            "p_champ": champ_p,
        }
    )

champ_df = pd.DataFrame(results).sort_values("p_champ", ascending=False)
print(champ_df[["short", "p_r2", "p_cf", "p_fin", "p_champ"]].to_string(index=False))

In [ ]:
BG = "#f7f7f7"
fig, ax = plt.subplots(figsize=(10, 5.5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

palette = [
    "#1a3a5c" if r["short"] == sn(champion[0]) else "#4a7ab5"
    for _, r in champ_df.iterrows()
]
bars = ax.bar(
    champ_df["short"],
    champ_df["p_champ"] * 100,
    color=palette,
    edgecolor="none",
    zorder=3,
)

for bar, (_, row) in zip(bars, champ_df.iterrows()):
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + 0.3,
        f"{h:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color="#1a1a1a",
    )

ax.set_ylabel("Championship probability (%)", fontsize=10)
ax.set_title(
    "2025–26 NBA Playoffs — Championship Probability\n"
    "(XGBoost · full bracket simulation over all opponent paths)",
    fontsize=11,
    pad=10,
)
ax.set_ylim(0, champ_df["p_champ"].max() * 100 * 1.18)
ax.yaxis.grid(True, alpha=0.4, color="#cccccc")
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="x", labelsize=10)

leg_handles = [
    mpatches.Patch(facecolor="#1a3a5c", label="Projected champion"),
    mpatches.Patch(facecolor="#4a7ab5", label="Other teams"),
]
ax.legend(handles=leg_handles, loc="upper right", fontsize=8.5, framealpha=0.7)

plt.tight_layout()
out = OUT_DIR / "championship_prob_2025-26.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {out}")

## 3. Model Performance Card

In [ ]:
from sklearn.metrics import auc as sk_auc
from sklearn.metrics import roc_curve

TEST_SEASONS = ["2022-23", "2023-24"]
TARGET_COL = "higher_seed_wins"

df = pd.read_parquet(PROC_DIR / "series_features.parquet")
train_df, test_df = train_test_split_by_season(df, TEST_SEASONS)
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]
X_train = train_df[FEATURE_COLS]

probs = pipeline.predict_proba(X_test)[:, 1]
preds = pipeline.predict(X_test)

# ROC
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = sk_auc(fpr, tpr)

# SHAP top features
scaler = pipeline.named_steps["scaler"]
clf = pipeline.named_steps["clf"]
X_train_sc = pd.DataFrame(scaler.transform(X_train), columns=FEATURE_COLS)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_COLS)
explainer = shap.TreeExplainer(clf)
shap_train = explainer.shap_values(X_train_sc)
mean_shap = pd.Series(np.abs(shap_train).mean(axis=0), index=FEATURE_COLS).sort_values(
    ascending=False
)

# Accuracy by round (test set)
round_acc = (
    test_df.assign(pred=preds)
    .groupby("round")
    .apply(lambda g: (g["pred"] == g[TARGET_COL]).mean())
)
round_names = {1: "R1", 2: "R2", 3: "Conf\nFinals", 4: "Finals"}

print(f"Test AUC: {roc_auc:.3f}")
print(f"Overall accuracy: {(preds == y_test).mean():.1%}")
print("By round:\n", round_acc.to_string())

In [ ]:
BG = "#f7f7f7"
fig = plt.figure(figsize=(15, 5.5))
fig.patch.set_facecolor(BG)

gs = fig.add_gridspec(1, 3, wspace=0.35)
ax_roc = fig.add_subplot(gs[0])
ax_shap = fig.add_subplot(gs[1])
ax_acc = fig.add_subplot(gs[2])
for ax in [ax_roc, ax_shap, ax_acc]:
    ax.set_facecolor(BG)

fig.suptitle(
    "Model Performance — XGBoost Series Predictor (holdout 2022–24)",
    fontsize=12,
    fontweight="bold",
    y=1.01,
)

# ── ROC curve ───────────────────────────────────────────────
ax_roc.plot(fpr, tpr, color="#1a3a5c", lw=2, label=f"AUC = {roc_auc:.3f}")
ax_roc.plot([0, 1], [0, 1], "--", color="#aaaaaa", lw=1)
ax_roc.fill_between(fpr, tpr, alpha=0.08, color="#1a3a5c")
ax_roc.set_xlabel("False Positive Rate", fontsize=9)
ax_roc.set_ylabel("True Positive Rate", fontsize=9)
ax_roc.set_title("ROC Curve", fontsize=10, pad=6)
ax_roc.legend(fontsize=9, loc="lower right")
ax_roc.spines[["top", "right"]].set_visible(False)
ax_roc.set_facecolor(BG)

# overall accuracy annotation
acc = (preds == y_test).mean()
ax_roc.text(
    0.97,
    0.08,
    f"Accuracy: {acc:.1%}\n(naive: 56.7%)",
    ha="right",
    va="bottom",
    fontsize=8,
    transform=ax_roc.transAxes,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", boxstyle="round"),
)

# ── SHAP feature importance ──────────────────────────────────
top_n = 8
top_feats = mean_shap.head(top_n)
FEAT_LABELS = {
    "ortg_diff": "OffRtg diff",
    "home_tov_pct": "Home TOV%",
    "away_win_pct": "Away W%",
    "away_net_rtg": "Away NetRtg",
    "away_ortg": "Away OffRtg",
    "home_drtg": "Home DefRtg",
    "away_pace": "Away Pace",
    "home_ortg": "Home OffRtg",
    "home_playoff_win_pct_3yr": "Home Playoff W%\n(3yr)",
    "net_rtg_diff": "NetRtg diff",
}
labels = [FEAT_LABELS.get(f, f) for f in top_feats.index]
colors_shap = ["#1a3a5c" if i < 3 else "#4a7ab5" for i in range(top_n)]
bars_shap = ax_shap.barh(
    labels[::-1], top_feats.values[::-1], color=colors_shap[::-1], edgecolor="none"
)
ax_shap.set_xlabel("Mean |SHAP value|", fontsize=9)
ax_shap.set_title("Top Features (SHAP)", fontsize=10, pad=6)
ax_shap.spines[["top", "right", "left"]].set_visible(False)
ax_shap.tick_params(axis="y", labelsize=8)
ax_shap.set_facecolor(BG)

# ── Accuracy by round ────────────────────────────────────────
rounds_present = sorted(round_acc.index)
rlabels = [round_names.get(r, str(r)) for r in rounds_present]
rvals = [round_acc[r] for r in rounds_present]
baseline = 0.567
bar_colors = ["#1a3a5c" if v >= baseline else "#d97b5b" for v in rvals]
ax_acc.bar(
    rlabels, [v * 100 for v in rvals], color=bar_colors, edgecolor="none", zorder=3
)
ax_acc.axhline(
    baseline * 100,
    color="#aaaaaa",
    lw=1.5,
    linestyle="--",
    label=f"Naive baseline ({baseline:.1%})",
    zorder=2,
)
for i, v in enumerate(rvals):
    ax_acc.text(
        i,
        v * 100 + 0.5,
        f"{v:.0%}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )
ax_acc.set_ylabel("Accuracy (%)", fontsize=9)
ax_acc.set_title("Accuracy by Round\n(2022–24 holdout)", fontsize=10, pad=6)
ax_acc.set_ylim(0, 105)
ax_acc.yaxis.grid(True, alpha=0.4, color="#cccccc")
ax_acc.set_axisbelow(True)
ax_acc.spines[["top", "right", "left"]].set_visible(False)
ax_acc.legend(fontsize=8, loc="lower right")
ax_acc.set_facecolor(BG)

plt.tight_layout()
out = OUT_DIR / "model_performance.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {out}")